# 17.7 Meta-learning and few-shot learning

The model learns how to adapt, so a new task begins with a useful bias instead of a blank slate.

Prototypical networks rebuild a classifier from support means. The same support-query geometry is evaluated as shots shrink and episode mismatch appears.

Save a copy to Drive to edit.

In [ ]:

import math
import warnings

import matplotlib.pyplot as plt
import numpy as np
from sklearn.base import clone
from sklearn.datasets import load_digits
from sklearn.datasets import load_iris
from sklearn.datasets import make_blobs
from sklearn.datasets import make_classification
from sklearn.datasets import make_moons
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings("ignore")
np.random.seed(17)

def budget_ladder():
    """Part 17 (learning paradigms): fix a real base dataset, shrink the LABEL budget per rung.

    Returns [(name, X, y, labeled_mask), ...] over the SAME digits data, only the fraction of
    labeled points falls D1->D5. The 'watch it scale' curve is accuracy vs label budget.
    """
    digits = load_digits()
    X = digits.data / 16.0
    y = digits.target
    rng = np.random.default_rng(17)
    rungs = []
    for name, frac in [("D1 100% labels", 1.0), ("D2 50% labels", 0.5), ("D3 20% labels", 0.2), ("D4 5% labels", 0.05), ("D5 2% labels", 0.02)]:
        mask = rng.random(y.shape) < frac
        if mask.sum() < 20:
            idx = rng.choice(len(y), size=20, replace=False)
            mask = np.zeros(len(y), dtype=bool)
            mask[idx] = True
        rungs.append((name, X, y, mask))
    return rungs


def split_labeled_train_test(X, y, labeled_mask, seed=0):
    train_idx, test_idx = train_test_split(
        np.arange(len(y)),
        test_size=0.35,
        random_state=seed,
        stratify=y,
    )
    labeled_idx = train_idx[labeled_mask[train_idx]]
    if len(np.unique(y[labeled_idx])) < len(np.unique(y)):
        rng = np.random.default_rng(seed)
        needed = []
        for cls in np.unique(y):
            choices = train_idx[y[train_idx] == cls]
            needed.append(rng.choice(choices))
        labeled_idx = np.unique(np.concatenate([labeled_idx, np.array(needed)]))
    return labeled_idx, train_idx, test_idx

def fit_logistic_accuracy(X, y, labeled_mask, seed=0):
    labeled_idx, train_idx, test_idx = split_labeled_train_test(X, y, labeled_mask, seed=seed)
    scaler = StandardScaler()
    scaler.fit(X[train_idx])
    X_labeled = scaler.transform(X[labeled_idx])
    X_test = scaler.transform(X[test_idx])
    clf = LogisticRegression(max_iter=1500, C=2.0, solver="lbfgs")
    clf.fit(X_labeled, y[labeled_idx])
    pred = clf.predict(X_test)
    return accuracy_score(y[test_idx], pred), clf, scaler, labeled_idx, test_idx

def ensure_class_budget_mask(y, frac, seed):
    rng = np.random.default_rng(seed)
    mask = np.zeros(len(y), dtype=bool)
    for cls in np.unique(y):
        cls_idx = np.where(y == cls)[0]
        count = max(1, int(round(frac * len(cls_idx))))
        chosen = rng.choice(cls_idx, size=count, replace=False)
        mask[chosen] = True
    return mask

def two_dimensional_view(X, seed=0):
    if X.shape[1] == 2:
        return X
    return PCA(n_components=2, random_state=seed).fit_transform(StandardScaler().fit_transform(X))

def plot_panel(ax, X, y, title, marked=None):
    Z = two_dimensional_view(X)
    ax.scatter(Z[:, 0], Z[:, 1], c=y, s=12, cmap="tab10", alpha=0.65)
    if marked is not None and len(marked) > 0:
        M = Z[marked]
        ax.scatter(M[:, 0], M[:, 1], facecolors="none", edgecolors="black", s=45, linewidths=1.0)
    ax.set_title(title, fontsize=9)
    ax.set_xticks([])
    ax.set_yticks([])


## The concept, built once

Prototypes $c_k=|S_k|^{-1}\sum_{x_i\in S_k}f_	heta(x_i)$ turn a support set into a classifier. The lesson query gets class-1 probability 0.652.

In [ ]:

def prototype_predict(support_X, support_y, query):
    labels = np.unique(support_y)
    prototypes = []
    for label in labels:
        prototypes.append(support_X[support_y == label].mean(axis=0))
    prototypes = np.vstack(prototypes)
    distances = ((prototypes - query) ** 2).sum(axis=1)
    weights = np.exp(-distances)
    probabilities = weights / weights.sum()
    return labels, prototypes, distances, probabilities

support_X = np.array([[-1.0, 0.0], [-0.8, 0.2], [1.0, 0.0], [0.9, -0.2]])
support_y = np.array([0, 0, 1, 1])
query = np.array([0.2, 0.05])
labels, prototypes, distances, probabilities = prototype_predict(support_X, support_y, query)

assert np.allclose(np.round(prototypes, 3), [[-0.900, 0.100], [0.950, -0.100]])
assert round(distances[0], 4) == 1.2125
assert round(distances[1], 4) == 0.5850
assert round(probabilities[1], 3) == 0.652
print("prototypes=", np.round(prototypes, 3))
print("distances=", np.round(distances, 4))
print(f"class-1 probability={probabilities[1]:.3f}")


The episodic evaluator repeatedly samples support and query sets, classifies queries by prototypes, and averages accuracy.

In [ ]:

def fewshot_episode_accuracy(X, y, shots=5, way=2, episodes=60, mismatch=False, seed=0):
    rng = np.random.default_rng(seed)
    scaler = StandardScaler()
    Xs = scaler.fit_transform(X)
    embed = PCA(n_components=16, random_state=seed).fit_transform(Xs)
    labels_all = np.unique(y)
    scores = []
    for episode in range(episodes):
        classes = rng.choice(labels_all, size=way, replace=False)
        support_X = []
        support_y = []
        query_X = []
        query_y = []
        for local_label, cls in enumerate(classes):
            idx = np.where(y == cls)[0]
            chosen = rng.choice(idx, size=shots + 8, replace=False)
            support = chosen[:shots]
            query = chosen[shots:]
            support_X.append(embed[support])
            support_y.append(np.full(shots, local_label))
            q_embed = embed[query].copy()
            if mismatch and episode % 2 == 0:
                q_embed = q_embed + rng.normal(0.0, 0.55, size=q_embed.shape)
            query_X.append(q_embed)
            query_y.append(np.full(len(query), local_label))
        support_X = np.vstack(support_X)
        support_y = np.concatenate(support_y)
        query_X = np.vstack(query_X)
        query_y = np.concatenate(query_y)
        labels = np.unique(support_y)
        prototypes = []
        for label in labels:
            prototypes.append(support_X[support_y == label].mean(axis=0))
        prototypes = np.vstack(prototypes)
        distances = ((query_X[:, None, :] - prototypes[None, :, :]) ** 2).sum(axis=2)
        pred = labels[np.argmin(distances, axis=1)]
        scores.append(accuracy_score(query_y, pred))
    return float(np.mean(scores))

def make_meta_ladder():
    digits = load_digits()
    X = digits.data / 16.0
    y = digits.target
    specs = [
        ("D1 5-shot matched", 5, 2, False),
        ("D2 5-shot clean", 5, 2, False),
        ("D3 1-shot noisy", 1, 2, True),
        ("D4 5-shot digits", 5, 5, False),
        ("D5 1-shot mismatch", 1, 5, True),
    ]
    return [(name, X, y, shots, way, mismatch) for name, shots, way, mismatch in specs]


## The dataset ladder

The rungs are few-shot episodes on real digits. D5 changes both shots and episode distribution, recreating the lesson's train/test mismatch warning.

In [ ]:

meta_rungs = make_meta_ladder()

for name, X, y, shots, way, mismatch in meta_rungs:
    print(f"{name:22s} X={X.shape} way={way} shots={shots} mismatch={mismatch}")


In [ ]:

meta_results = []

for name, X, y, shots, way, mismatch in meta_rungs:
    acc = fewshot_episode_accuracy(X, y, shots=shots, way=way, mismatch=mismatch, seed=9)
    meta_results.append((name, shots, way, mismatch, acc, X, y))
    print(f"{name:22s} episodic_accuracy={acc:.3f}")


In [ ]:

fig, axes = plt.subplots(2, 5, figsize=(14, 5))

for ax, row in zip(axes[0], meta_results):
    name, shots, way, mismatch, acc, X, y = row
    plot_panel(ax, X, y, name.split()[0] + f" acc={acc:.2f}")

shots_values = [row[1] for row in meta_results]
accuracies = [row[4] for row in meta_results]
axes[1, 0].plot(shots_values, accuracies, marker="o")
axes[1, 0].invert_xaxis()
axes[1, 0].set_xlabel("shots")
axes[1, 0].set_ylabel("episodic accuracy")
axes[1, 0].set_title("accuracy vs shots/task shift")

for ax in axes[1, 1:]:
    ax.axis("off")

plt.tight_layout()
plt.show()


## Pitfall on D5: mismatched episodes

Training or validation episodes that are cleaner than deployment overstate few-shot readiness. Matching way, shot, and noise makes the estimate honest.

In [ ]:

digits = load_digits()
X = digits.data / 16.0
y = digits.target
clean_episode_acc = fewshot_episode_accuracy(X, y, shots=5, way=2, mismatch=False, seed=9)
mismatch_acc = fewshot_episode_accuracy(X, y, shots=1, way=5, mismatch=True, seed=9)
matched_acc = fewshot_episode_accuracy(X, y, shots=1, way=5, mismatch=False, seed=9)

print(f"clean validation episodes: {clean_episode_acc:.3f}")
print(f"D5 mismatched deployment: {mismatch_acc:.3f}")
print(f"D5 matched sampler estimate: {matched_acc:.3f}")
assert matched_acc >= mismatch_acc - 0.10


## Evaluate it + Practice

- Metric: episodic accuracy vs shots/task shift, always compared with a no-skill majority or scratch baseline.
- Sanity check: labels must be shuffled or withheld to confirm the method loses its advantage.
- Ablation: turn off the key paradigm idea and verify the metric drops.
- Failure signal: the diagnostic score improves while held-out target accuracy falls.
- Robustness check: repeat with a different seed and inspect the hardest rung.

### Practice 1
Change the budget or shift on D4 and rerun the summary curve.

### Practice 2
Add one ablation that removes the paradigm-specific step.

### Practice 3
Write a one-sentence deployment warning for D5.